In [ ]:
import random, numpy, torch

SEED = 42
random.seed(SEED)
numpy.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

from mircdatasethandler import data_utils

root_dataset_folder = "../mirc-dataset/annotations/single-person/"
dataloaders = data_utils.create_dataloaders_pipeline(root_dataset_folder, k=5, batch_size=4)

In [ ]:
#examples for using API

#from elmprune import ELMImportanceProcessor, ImportanceProcessorConfig
#from elmprune.utils import get_model

# model = models_utils.get_model("Unet", "resnet34")
#dataloader = dataloaders[0][1]

#elm_importance_processor = ELMImportanceProcessor(ImportanceProcessorConfig(), model, dataloader)
#layerwise_importances = elm_importance_processor.compute_elm_layerwise_importances()
#filterwise_importances = elm_importance_processor.compute_elm_filterwise_importances()
#global_importances = elm_importance_processor.compute_elm_global_importances()

# from elmprune import PruneProcessor
# result = {}
# percentages_default = [0.2, 0.4, 0.6, 0.8]
# for p in percentages_default:
#     prune_processor = PruneProcessor(model, importances, p, dataloader)
#     prunned_model = prune_processor.execute()
#     result[f"{p}"] = sum(p.numel() for p in prunned_model.parameters())
# 
#  
# print(f"percentage 1 - {sum(p.numel() for p in model.parameters())} params")
# for p in result.keys():
#     print(f"percentage {p} - {result[p]} params")

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
from elmprune.utils import find_best_val_loss_files, mirror_copy_files, collect_infos, get_and_load_model
from elmprune.utils import get_first_dataloader_image, save_pruned_model, get_val_dataloader_fold
from elmprune import ELMImportanceProcessor, ImportanceProcessorConfig
from elmprune import PruneProcessor

class PrunePipeline:

    pruned_suffix = "-pruned"
    prune_percentages_default = [0.2, 0.4, 0.6, 0.8]
    supported_prune_types = ["elm", "mag", "random", "tests"]

    def __init__(self, dataloaders, models_root_path, task, subset, prune_percentages = None, prune_mode = 'elm'):
        self.src_root = Path(models_root_path) / task / subset
        self.dst_root = Path(models_root_path) / (task+self.pruned_suffix) / subset
        self.src_files = find_best_val_loss_files(self.src_root)
        self.prune_percentages = self.prune_percentages_default if prune_percentages is None else prune_percentages
        self.dataloaders = dataloaders
        self.prune_mode = prune_mode
        if prune_mode not in self.supported_prune_types:
            raise Exception("Prune type currently not supported!") 
        if prune_mode in ["mag", "random"]:
            raise Exception(f"Prune type {prune_mode} will be implemented in future!")

    def execute(self):
        mirror_copy_files(self.src_root, self.dst_root)
        dst_infos = collect_infos(self.dst_root)
        
        for dst_file_model in tqdm(dst_infos, desc="Pruning runner"):
            model = get_and_load_model(dst_file_model["model"], dst_file_model["backbone"], dst_file_model["full_path"])
            dataloader = get_val_dataloader_fold(self.dataloaders, dst_file_model["fold"])

            if self.prune_mode == 'elm':
                importances = self.__get_importances_elm_prune(model, dataloader)
            elif self.prune_mode == "tests":
                importances = self.__get_importances_tests()
            
            for importance_type, importances in zip(importances.keys(), importances.values()):
                for prune_percentage in tqdm(self.prune_percentages, desc="Pruning percentage"):
                    input_example = get_first_dataloader_image(dataloader)
                    prune_processor = PruneProcessor(model, importances, prune_percentage, input_example, verbose = False)
                    pruned_model = prune_processor.execute()
                    path_out = Path(dst_file_model["full_path"]).resolve().parent
                    save_pruned_model(pruned_model, input_example, path_out, importance_type)

    def __get_importances_elm_prune(self, model, dataloader):
        print("Extracting features for ELM...")
        elm_importance_processor = ELMImportanceProcessor(ImportanceProcessorConfig(), model, dataloader)
        print("Getting importances for layerwise...")
        layerwise_importances = elm_importance_processor.compute_elm_layerwise_importances()
        print("Getting importances for filterwise...")
        filterwise_importances = elm_importance_processor.compute_elm_filterwise_importances()
        print("Getting importances for global...")
        global_importances = elm_importance_processor.compute_elm_global_importances()
        return  {
                    "elm_layerwise": layerwise_importances, 
                    "elm_filterwise": filterwise_importances, 
                    "elm_global": global_importances
                }
    
    def __get_importances_tests(self):
        from elmprune.utils import load_dict
        print("Getting importances for test...")
        importances = load_dict(Path("test_dict.json"))
        return  {
                    "tests": importances,
                }

In [15]:
models_root_path = '../../../assets/doctorate/training-models/'
task = "segmentation"
subset = "single-person" #pass as param to runner

prune_pipeline = PrunePipeline(dataloaders, models_root_path, task, subset, prune_mode="car")
prune_pipeline.execute()

Prune type currently not supported!


RuntimeError: No active exception to reraise